In [2]:
import pandas as pd
import numpy as np

# display more column
pd.set_option("display.max_columns", 50)


1. Data Loading and Initial Inspection

In this step, we load the training and test datasets (epl-training.csv and epl-test.csv) into Pandas DataFrames.
This allows us to examine the basic structure of the data, including the number of rows, columns, and the general format of the features.


In [16]:
# read training testing set
epl_train = pd.read_csv("epl-training.csv")
epl_test  = pd.read_csv("epl-test.csv")

train_df = epl_train.copy()
test_df  = epl_test.copy()

print("Train shape:", train_df.shape)
print("Test shape:",  test_df.shape)

train_df.head()


Train shape: (9601, 22)
Test shape: (10, 3)


,Date,HomeTeam,AwayTeam,FTHG,FTAG,FTR,HTHG,HTAG,HTR,Referee,HS,AS,HST,AST,HC,AC,HF,AF,HY,AY,HR,AR
0,19/08/2000,Charlton,Man City,4.0,0.0,H,2.0,0.0,H,Rob Harris,17.0,8.0,14.0,4.0,6.0,6.0,13.0,12.0,1.0,2.0,0.0,0.0
1,19/08/2000,Chelsea,West Ham,4.0,2.0,H,1.0,0.0,H,Graham Barber,17.0,12.0,10.0,5.0,7.0,7.0,19.0,14.0,1.0,2.0,0.0,0.0
2,19/08/2000,Coventry,Middlesbrough,1.0,3.0,A,1.0,1.0,D,Barry Knight,6.0,16.0,3.0,9.0,8.0,4.0,15.0,21.0,5.0,3.0,1.0,0.0
3,19/08/2000,Derby,Southampton,2.0,2.0,D,1.0,2.0,A,Andy D'Urso,6.0,13.0,4.0,6.0,5.0,8.0,11.0,13.0,1.0,1.0,0.0,0.0
4,19/08/2000,Leeds,Everton,2.0,0.0,H,2.0,0.0,H,Dermot Gallagher,17.0,12.0,8.0,6.0,6.0,4.0,21.0,20.0,1.0,3.0,0.0,0.0


2. Missing Value Analysis

Before performing any preprocessing or feature engineering, we need to determine whether the dataset contains missing values.
Missing values can interfere with model training, cause errors during feature engineering, or reduce predictive performance.

In [5]:
print("Missing values in train:\n", train_df.isna().sum())
print("\nMissing values in test:\n",  test_df.isna().sum())


Missing values in train:
 Date        1
HomeTeam    1
AwayTeam    1
FTHG        1
FTAG        1
FTR         1
HTHG        1
HTAG        1
HTR         1
Referee     1
HS          1
AS          1
HST         1
AST         1
HC          1
AC          1
HF          1
AF          1
HY          1
AY          1
HR          1
AR          1
dtype: int64

Missing values in test:
 Date        0
HomeTeam    0
AwayTeam    0
dtype: int64


In [15]:

print("Missing values in train BEFORE dropna:")
print(train.isnull().sum())

# —— delete NaN column，only affect train，
train = train.dropna().reset_index(drop=True)

print("\nMissing values in train AFTER dropna:")
print(train.isnull().sum())
print("Train shape after dropna:", train.shape)

print("\nMissing values in test:")
print(test.isnull().sum())
print("Test shape:", test.shape)


Missing values in train BEFORE dropna:
Date        0
HomeTeam    0
AwayTeam    0
FTHG        0
FTAG        0
FTR         0
HTHG        0
HTAG        0
HTR         0
Referee     0
HS          0
AS          0
HST         0
AST         0
HC          0
AC          0
HF          0
AF          0
HY          0
AY          0
HR          0
AR          0
dtype: int64

Missing values in train AFTER dropna:
Date        0
HomeTeam    0
AwayTeam    0
FTHG        0
FTAG        0
FTR         0
HTHG        0
HTAG        0
HTR         0
Referee     0
HS          0
AS          0
HST         0
AST         0
HC          0
AC          0
HF          0
AF          0
HY          0
AY          0
HR          0
AR          0
dtype: int64
Train shape after dropna: (9600, 22)

Missing values in test:
Date        0
HomeTeam    0
AwayTeam    0
dtype: int64
Test shape: (10, 3)


Step 3 — Date Parsing & Missing-Value Cleaning 
In this step, we convert the Date column in both the training and test datasets into a proper datetime format. The training file uses the format “19/08/2000” (day/month/year), while the test file uses a different style “31 Jan 26” (day + month abbreviation + two-digit year).
Since machine-learning models cannot interpret raw text dates, converting them into standard datetime objects is essential for later steps such as sorting matches chronologically (required for ELO computation).

We also clean any rows where the date cannot be parsed, because those rows would cause errors in downstream processing.

In [12]:
# ===== Step 3: Parse dates properly and clean missing rows =====

import pandas as pd

# training set 
train["Date"] = pd.to_datetime(train["Date"], format="%d/%m/%Y", errors="coerce")

# test set form"31 Jan 26" → day + month abbreviation + 2-digit year
test["Date"] = pd.to_datetime(test["Date"], format="%d %b %y", errors="coerce")

print("Train date NA:", train["Date"].isna().sum())
print("Test date NA:", test["Date"].isna().sum())

# delete missing value in test set 
train = train.dropna().reset_index(drop=True)

print("After cleaning → Train shape:", train.shape)
print("After cleaning → Test shape:", test.shape)
# ===== Step 3: Parse dates properly and clean missing rows =====

import pandas as pd

# training set form
train["Date"] = pd.to_datetime(train["Date"], format="%d/%m/%Y", errors="coerce")

# test set"31 Jan 26" → day + month abbreviation + 2-digit year
test["Date"] = pd.to_datetime(test["Date"], format="%d %b %y", errors="coerce")

# check if there is any unrecognised date
print("Train date NA:", train["Date"].isna().sum())
print("Test date NA:", test["Date"].isna().sum())

# deleting training set missing value
train = train.dropna().reset_index(drop=True)

print("After cleaning → Train shape:", train.shape)
print("After cleaning → Test shape:", test.shape)


Train date NA: 0
Test date NA: 0
After cleaning → Train shape: (9600, 22)
After cleaning → Test shape: (10, 3)
Train date NA: 0
Test date NA: 0
After cleaning → Train shape: (9600, 22)
After cleaning → Test shape: (10, 3)


Constructing Hard-Difficulty Features Using an ELO Rating System

In this step, I implement an ELO rating algorithm to generate advanced, high-level features for each match.
ELO ratings are widely used in competitive games (e.g., chess, esports, and sports) to evaluate the relative strength of competing players or teams.

In [13]:
def add_elo_features(
    df: pd.DataFrame,
    base_rating: float = 1500.0,
    k_factor: float = 20.0,
    home_advantage: float = 50.0,
    initial_ratings: dict | None = None
):
    """
    Compute ELO ratings over time and attach, for each match:
      - home_elo_pre: rating of HomeTeam *before* this match
      - away_elo_pre: rating of AwayTeam *before* this match
      - elo_diff_pre: home_elo_pre - away_elo_pre

    Parameters
    ----------
    df : DataFrame with at least ['Date','HomeTeam','AwayTeam'] and (for training) 'FTR'
    base_rating : starting ELO for any team we haven't seen yet
    k_factor : learning rate for rating updates
    home_advantage : rating points added to home team when computing expectation
    initial_ratings : optional dict {team_name: elo} to start from
                      (we will use this when we process the TEST set)
    """
    df = df.copy()
    df = df.sort_values("Date").reset_index(drop=True)

    # each team ELO value
    ratings = dict(initial_ratings) if initial_ratings is not None else {}

    home_elo_pre = []
    away_elo_pre = []

    def result_to_scores(ftr: str):
        """Map FTR (H/D/A) to numeric scores for ELO update."""
        if ftr == "H":
            return 1.0, 0.0
        elif ftr == "A":
            return 0.0, 1.0
        elif ftr == "D":
            return 0.5, 0.5
        else:
            # if no result no update
            return None, None

    has_result_col = "FTR" in df.columns

    for _, row in df.iterrows():
        home = row["HomeTeam"]
        away = row["AwayTeam"]

        # if first time appear give a basic mark
        R_h = ratings.get(home, base_rating)
        R_a = ratings.get(away, base_rating)

        # record elo before match
        home_elo_pre.append(R_h)
        away_elo_pre.append(R_a)

        # test set has no FTR，jump through pre-rating
        if not has_result_col:
            continue

        ftr = row["FTR"]
        S_h, S_a = result_to_scores(ftr)
        if S_h is None:
            continue  # no result not update

        # 
        exp_home = 1.0 / (1.0 + 10 ** (-(R_h + home_advantage - R_a) / 400.0))
        exp_away = 1.0 - exp_home

        # update ELO
        R_h_new = R_h + k_factor * (S_h - exp_home)
        R_a_new = R_a + k_factor * (S_a - exp_away)

        ratings[home] = R_h_new
        ratings[away] = R_a_new

    #  pre-match ELO back to DataFrame
    df["home_elo_pre"] = home_elo_pre
    df["away_elo_pre"] = away_elo_pre
    df["elo_diff_pre"] = df["home_elo_pre"] - df["away_elo_pre"]

    return df, ratings


Step 5 — Apply ELO to Train and Test
Applying ELO Features to Training and Test Sets

In this step, I use the add_elo_features() function to generate hard-difficulty ELO-based features for both datasets.

In [14]:
# 1) calculate ELO on training data
train_with_elo, final_elo_ratings = add_elo_features(train)

print("Sample of training data with ELO features:")
display(
    train_with_elo[["Date", "HomeTeam", "AwayTeam", "FTR",
                    "home_elo_pre", "away_elo_pre", "elo_diff_pre"]].head()
)

# 2) final ELO from training set，create ELO feature on test set
test_with_elo, _ = add_elo_features(
    test,
    initial_ratings=final_elo_ratings  # from last round mark based on training set
)

print("\nSample of test data with ELO features:")
display(
    test_with_elo[["Date", "HomeTeam", "AwayTeam",
                   "home_elo_pre", "away_elo_pre", "elo_diff_pre"]].head()
)


Sample of training data with ELO features:


,Date,HomeTeam,AwayTeam,FTR,home_elo_pre,away_elo_pre,elo_diff_pre
0,2000-08-19,Charlton,Man City,H,1500.0,1500.0,0.0
1,2000-08-19,Chelsea,West Ham,H,1500.0,1500.0,0.0
2,2000-08-19,Coventry,Middlesbrough,A,1500.0,1500.0,0.0
3,2000-08-19,Derby,Southampton,D,1500.0,1500.0,0.0
4,2000-08-19,Leeds,Everton,H,1500.0,1500.0,0.0



Sample of test data with ELO features:


,Date,HomeTeam,AwayTeam,home_elo_pre,away_elo_pre,elo_diff_pre
0,2026-01-31,Leeds,Arsenal,1461.904280,1774.596001,-312.691721
1,2026-01-31,Liverpool,Newcastle,1786.326629,1662.949833,123.376796
2,2026-01-31,Tottenham,Man City,1534.176660,1782.688444,-248.511785
3,2026-01-31,Wolves,Bournemouth,1535.761463,1582.414687,-46.653224
4,2026-01-31,Aston Villa,Brentford,1682.771184,1594.926932,87.844252
